# 6.5 CrewAI 框架入门

> **AirSim 配置**：本节不需要 AirSim 模拟器，纯 CrewAI 框架演示。
>
> **Python 环境**：需要 Python 3.10+，使用 `conda activate drone` 环境。

前面我们用纯Python实现了中心化和分布式两种协同模式。在实际工程中，通常会使用多Agent框架来简化开发。

本节介绍 **CrewAI** —— 学习曲线最低的多Agent框架，只需掌握3个核心概念即可上手。

## CrewAI 是什么？

CrewAI 将多Agent协作类比为**现实中的工作团队**。你只需要定义：

| 概念 | 类比 | 作用 |
|------|------|------|
| **Agent** | 团队成员 | 定义角色（role）、目标（goal）、背景（backstory） |
| **Task** | 工作任务 | 定义任务描述和期望输出，指定由哪个Agent执行 |
| **Crew** | 团队 | 把Agent和Task组合在一起，启动执行 |

```
┌─────────── Crew（团队）───────────┐
│                                    │
│  Agent1（研究员）→ Task1（调研）    │
│         ↓                          │
│  Agent2（写手）  → Task2（撰写）    │
│                                    │
└────────────────────────────────────┘
          crew.kickoff() → 结果
```

就这么简单——定义角色、分配任务、组建团队、开始工作。

![crewai.png](img/crewai.png)

## 安装

In [ ]:
#!pip install crewai

## 配置 LLM

CrewAI 内置了 `LLM` 类，可以直接连接任何 OpenAI 兼容的 API。我们用豆包（Doubao）模型：

In [1]:
from crewai import LLM

# 配置豆包模型（兼容 OpenAI 接口）
llm = LLM(
    model="doubao-seed-2-0-pro-260215",   # 替换为你的模型 ID
    base_url="https://ark.cn-beijing.volces.com/api/v3",
    api_key="",
    temperature=0.1
)

print("LLM 配置完成")

11:17:03 - LiteLLM:WARNING: get_model_cost_map.py:271 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: _ssl.c:1000: The handshake operation timed out. Falling back to local backup.


LLM 配置完成


## 示例1：单Agent Hello World

最简单的 CrewAI 程序：一个Agent、一个Task、一个Crew。

In [2]:
from crewai import Agent, Task, Crew

# 1. 定义 Agent（角色）
greeter = Agent(
    role="问候专家",
    goal="用有趣的方式向用户打招呼",
    backstory="你是一个热情友好的AI助手，擅长用创意方式打招呼。",
    llm=llm,
    verbose=True
)

# 2. 定义 Task（任务）
greet_task = Task(
    description="请用中文向'小明'打招呼，并给出今天的一句鼓励话语。",
    expected_output="一段友好的中文问候语，包含鼓励话语。",
    agent=greeter
)

# 3. 组建 Crew（团队）并执行
crew = Crew(
    agents=[greeter],
    tasks=[greet_task],
    verbose=True
)

result = crew.kickoff()
print("\n===== 结果 =====")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 89ba91c8-392e-45fd-828f-88fd6b146df8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 请用中文向'小明'打招呼，并给出今天的一句鼓励话语。                                                       │
│  ID: 37c95ef6-3cbd-4461-ba8e-01a1b975bac1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 问候专家                                                                                                │
│                                                                                                                 │
│  Task: 请用中文向'小明'打招呼，并给出今天的一句鼓励话语。                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 问候专家                                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  嗨小明，今天好呀！😉                                                                                           │
│  给你送上今日专属鼓励哦：你认真对待每件事的样子超棒的，哪怕遇到点小磕小绊也没关系，所有你悄悄付出的努力都在偷   │
│  偷给你攒惊喜呢，今天也只管放开手脚去做想做的事就好，你可比你想象中厉害多啦，加油！                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 请用中文向'小明'打招呼，并给出今天的一句鼓励话语。                                                       │
│  Agent: 问候专家                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== 结果 =====
嗨小明，今天好呀！😉
给你送上今日专属鼓励哦：你认真对待每件事的样子超棒的，哪怕遇到点小磕小绊也没关系，所有你悄悄付出的努力都在偷偷给你攒惊喜呢，今天也只管放开手脚去做想做的事就好，你可比你想象中厉害多啦，加油！


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> 就这么简单！定义 Agent → 定义 Task → 组建 Crew → `kickoff()` 执行。

## 示例2：双Agent协作

两个Agent按顺序协作：**研究员**先收集信息，**报告员**再整理成报告。

这就是 CrewAI 的核心价值——用角色扮演的方式编排多Agent协作。

In [3]:
# Agent 1：研究员
researcher = Agent(
    role="无人机技术研究员",
    goal="收集和整理无人机集群协同的关键技术要点",
    backstory="你是一位资深的无人机技术专家，熟悉集群协同、路径规划和通信技术。",
    llm=llm,
    verbose=True
)

# Agent 2：报告员
reporter = Agent(
    role="技术报告撰写员",
    goal="将研究员的发现整理成简洁易懂的中文报告",
    backstory="你是一位技术写手，擅长将复杂技术概念用通俗语言表达。",
    llm=llm,
    verbose=True
)

# Task 1：研究任务
research_task = Task(
    description="列出无人机集群协同的3个核心技术挑战，每个挑战用2-3句话说明。",
    expected_output="3个核心技术挑战的列表，每个包含简要说明。",
    agent=researcher
)

# Task 2：报告任务（基于研究结果）
report_task = Task(
    description="根据研究员提供的技术要点，撰写一段200字以内的中文技术简报。",
    expected_output="一段简洁的中文技术简报，200字以内。",
    agent=reporter
)

# 组建团队（顺序执行：先研究，再撰写）
crew = Crew(
    agents=[researcher, reporter],
    tasks=[research_task, report_task],
    verbose=True
)

result = crew.kickoff()
print("\n===== 最终报告 =====")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b0d27dbb-d6ec-42d2-8e57-aafab974e140                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 89ba91c8-392e-45fd-828f-88fd6b146df8                                                                       │
│  Final Output: 嗨小明，今天好呀！😉                                                                             │
│  给你送上今日专属鼓励哦：你认真对待每件事的样子超棒的，哪怕遇到点小磕小绊也没关系，所有你悄悄付出的努力都在偷   │
│  偷给你攒惊喜呢，今天也只管放开手脚去做想做的事就好，你可比你想象中厉害多啦，加油！                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 列出无人机集群协同的3个核心技术挑战，每个挑战用2-3句话说明。                                             │
│  ID: b2ed8340-607d-4964-b76c-0bea8acc10e4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 无人机技术研究员                                                                                        │
│                                                                                                                 │
│  Task: 列出无人机集群协同的3个核心技术挑战，每个挑战用2-3句话说明。                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 无人机技术研究员                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### 无人机集群协同3个核心技术挑战                                                                              │
│  1. **分布式协同感知与态势共识挑战**                                                                            │
│  无人机个体搭载的传感器存在视场、精度局限性，且易受环境干扰产生观测偏差、数据缺失问题，需要在通信存在延迟、丢   │
│  包的非理想条件下完成多源异构观测数据的时空对齐与融合，实现全集群对作业环境、任务目标的一致认知。态势共识是集   │
│  群开展所有协同动作的基础前提，一旦出现认知偏差，极易引发任务失败、集群内部碰撞等严重问题。                     │
│                                                                                                                 │
│  2. **动态复杂环境下的分布式路径规划与避碰挑战**                                                                │
│  无人机集群的作业场景往往存在动态障碍物、临时禁飞区等不确定要素，且集群节点自身处于高速运动状态，运动耦合关系   │
│  极为复杂，传统集中式路径规划存在算力压力过大、单点故障导致全集群瘫痪的风险，分布式规划模式则需要兼顾全局任务   │
│  效率与个体避碰需求，二者的平衡难度极高。大规模集群还要求规划算法具备毫秒级响应速度，同时要规避路径冲突、运动   │
│  死锁等问题，对算法的实时性、鲁棒性提出了严苛要求。                                                             │
│                                                                                                                 │
│  3. **强约束下的自主通信组网挑战**                                                                              │
│  无人机集群节点高速移动会导致网络拓扑频繁动态变化，作业场景的电磁遮挡、干扰也极易引发通信链路中断，而集群协同   │
│  需要态势数据、控制指令的传输满足低时延、高可靠要求，传统固定参数的通信组网协议无法适配这种动态性极强的网络环   │
│  境。同时集群可用通信带宽资源有限，需要实现高优先级任务数据的优先调度，还要避免大量节点同时传输造成的信道拥塞   │
│  ，是维持集群协同控制闭环的核心支撑技术。                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 技术报告撰写员                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### 无人机集群协同核心技术挑战简报                                                                             │
│  当前无人机集群协同落地面临三大核心技术瓶颈：一是分布式协同感知与态势共识，需在非理想通信条件下完成多源数据融   │
│  合，形成全局一致认知，是协同动作的基础；二是动态环境下分布式路径规划与避碰，需平衡全局效率与个体安全，满足毫   │
│  秒级响应要求；三是强约束下自主通信组网，需适配高速移动带来的拓扑动态变化，保障低时延高可靠传输，规避信道拥塞   │
│  。                                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 根据研究员提供的技术要点，撰写一段200字以内的中文技术简报。                                              │
│  Agent: 技术报告撰写员                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== 最终报告 =====
### 无人机集群协同核心技术挑战简报
当前无人机集群协同落地面临三大核心技术瓶颈：一是分布式协同感知与态势共识，需在非理想通信条件下完成多源数据融合，形成全局一致认知，是协同动作的基础；二是动态环境下分布式路径规划与避碰，需平衡全局效率与个体安全，满足毫秒级响应要求；三是强约束下自主通信组网，需适配高速移动带来的拓扑动态变化，保障低时延高可靠传输，规避信道拥塞。


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b0d27dbb-d6ec-42d2-8e57-aafab974e140                                                                       │
│  Final Output: ### 无人机集群协同核心技术挑战简报                                                               │
│  当前无人机集群协同落地面临三大核心技术瓶颈：一是分布式协同感知与态势共识，需在非理想通信条件下完成多源数据融   │
│  合，形成全局一致认知，是协同动作的基础；二是动态环境下分布式路径规划与避碰，需平衡全局效率与个体安全，满足毫   │
│  秒级响应要求；三是强约束下自主通信组网，需适配高速移动带来的拓扑动态变化，保障低时延高可靠传输，规避信道拥塞   │
│  。                                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> 注意观察执行日志：Task1（研究）先执行，其输出自动传递给 Task2（撰写）作为上下文。这就是 CrewAI 的顺序协作模式。

## 自定义工具（@tool）

Agent 不仅能"思考"，还能"行动"——通过**工具（Tool）**与外部世界交互。

CrewAI 提供了 `@tool` 装饰器，可以将任意 Python 函数变成 Agent 可调用的工具：

In [4]:
from crewai.tools import tool

# 定义一个模拟天气查询工具
# 注意：@tool 的名称必须用英文，中文会被 CrewAI 过滤为空字符串导致报错
@tool("get_weather")
def get_weather(city: str) -> str:
    """查询指定城市的天气情况（模拟数据）。"""
    weather_data = {"北京": "晴天 25°C", "上海": "下雨 20°C", "广州": "多云 30°C"}
    return weather_data.get(city, f"{city}: 暂无数据")

# 定义一个模拟穿衣建议工具
@tool("suggest_clothing")
def suggest_clothing(weather: str) -> str:
    """根据天气情况给出穿衣建议。"""
    if "下雨" in weather:
        return "建议带伞，穿防水外套"
    elif "30" in weather:
        return "天气炎热，穿短袖防晒"
    else:
        return "天气舒适，穿长袖即可"

# 创建一个带工具的 Agent
weather_agent = Agent(
    role="天气助手",
    goal="查询天气并给出穿衣建议",
    backstory="你是一个贴心的天气助手，会先查天气再给建议。",
    tools=[get_weather, suggest_clothing],
    llm=llm,
    verbose=True
)

weather_task = Task(
    description="查询北京今天的天气，并给出穿衣建议。",
    expected_output="天气情况和对应的穿衣建议。",
    agent=weather_agent
)

crew = Crew(agents=[weather_agent], tasks=[weather_task], verbose=True)
result = crew.kickoff()
print("\n===== 结果 =====")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 97c9a900-ebb2-46a2-9445-c7e2847bcac3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 查询北京今天的天气，并给出穿衣建议。                                                                     │
│  ID: 186a28bb-0b65-4ea5-a77a-b4f872879bc3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 天气助手                                                                                                │
│                                                                                                                 │
│  Task: 查询北京今天的天气，并给出穿衣建议。                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool get_weather executed with result: 晴天 25°C...

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: get_weather                                                                                              │
│  Args: {'city': '北京'}                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 天气助手                                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  北京今天的天气为晴天，气温25°C，穿衣建议：天气舒适，穿长袖即可。                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 查询北京今天的天气，并给出穿衣建议。                                                                     │
│  Agent: 天气助手                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== 结果 =====
北京今天的天气为晴天，气温25°C，穿衣建议：天气舒适，穿长袖即可。


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 97c9a900-ebb2-46a2-9445-c7e2847bcac3                                                                       │
│  Final Output: 北京今天的天气为晴天，气温25°C，穿衣建议：天气舒适，穿长袖即可。                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> Agent 会自动决定何时调用哪个工具——这就是 **ReAct（推理+行动）** 模式的体现。
> LLM 先"思考"需要什么信息，再"行动"调用工具获取，最后整合输出。

## 总结

CrewAI 的核心就3个概念：

```python
Agent(role, goal, backstory, tools, llm)  # 定义角色
Task(description, expected_output, agent)  # 分配任务
Crew(agents, tasks).kickoff()              # 组建团队、开始工作
```

| 对比 | 纯Python（笔记本3-4） | CrewAI（本节） |
|------|----------------------|----------------|
| 任务分配 | 手动写代码 | 框架自动编排 |
| 工具调用 | 手动调用函数 | Agent自动决定何时调用 |
| 多Agent协作 | 手动传递结果 | 框架自动传递上下文 |
| 代码量 | 少但需自己管理流程 | 更少且框架管理流程 |

下一节，我们将用 CrewAI 的 `@tool` 机制封装 AirSim API，让 Agent 直接控制无人机。